<a href="https://colab.research.google.com/github/Thedreamaker/Qiskit_Learning/blob/basics/basics_phase_estimation/quantum_phase_estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.7 MB/s eta 0:00:00


In [11]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# Create the circuit
qpe_fixed = QuantumCircuit(3, 2)

# 1. Target preparation
qpe_fixed.x(2)

# 2. Open superpositions
qpe_fixed.h(0)
qpe_fixed.h(1)

# 3. Phase encoding
qpe_fixed.cp(np.pi / 2, 0, 2)
qpe_fixed.cz(1, 2)

qpe_fixed.barrier()

# 4. YOUR FIXED IQFT LAYER! (Using -pi/4 to perfectly erase the leak)
qpe_fixed.h(0)
qpe_fixed.cp(-np.pi / 2, 0, 1)  # <-- Changed from -pi/2 to your correct -pi/4!
qpe_fixed.h(1)

# 5. Measure
qpe_fixed.measure([0, 1], [0, 1])

# 6. Run 1000 times to test for true certainty
sim = AerSimulator()
result = sim.run(qpe_fixed, shots=1000).result().get_counts()
print("Fixed QPE Results (1000 shots):")
print(result)


Fixed QPE Results (1000 shots):
{'01': 253, '11': 243, '10': 504}


In [17]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# Qubits 0 and 1 are 'counting qubits' (readout). Qubit 2 holds the target state.
qpe_qc = QuantumCircuit(3, 2)

# 1. Prepare the target qubit (Qubit 2) in state |1>
qpe_qc.x(2)

# 2. Put counting qubits into superposition
qpe_qc.h(0)
qpe_qc.h(1)

# 3. Controlled-Phase Operations (This encodes the secret rhythm/phase)
# We apply a secret phase of 1/4 of a rotation (pi/2 radians)
qpe_qc.cp(np.pi / 2, 0, 2)
qpe_qc.cp(np.pi, 1, 2) # Doubling the angle for the next power of 2

qpe_qc.barrier()

# 4. The Inverse QFT (Swaps the wave phase back into readable bits)
qpe_qc.h(0)
qpe_qc.cp(-np.pi / 2, 0, 1)
qpe_qc.h(1)

# 5. Measure the counting qubits
qpe_qc.measure([0, 1], [0, 1])

# 6. Simulate
sim = AerSimulator()
result = sim.run(qpe_qc, shots=1000).result().get_counts()
print("QPE Measuring Outcome (Binary phase representation):")
print(result)
# '01' in binary translates to 1/4 (0.01 in binary fraction), matching our pi/2 rotation!


QPE Measuring Outcome (Binary phase representation):
{'11': 273, '10': 508, '01': 219}
